# 01 — Data Inspection

This notebook answers one question: **What is actually inside our frozen StatsBomb V1 dataset?**

We will inspect the raw data without cleaning or changing it. The observations here will determine the feature engineering we do later.

## Step 0 — Understand the layers

- **Matches** describe games and context.
- **Lineups** describe which players were involved and their position intervals.
- **Events** describe football actions such as passes, shots and carries.
- **360** is optional and is not required by the V1 core model.

StatsBomb documents this JSON organization in its Open Data repository.

In [ ]:
from pathlib import Path
import json
from collections import Counter
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "statsbomb"
MANIFEST_PATH = PROJECT_ROOT / "data" / "dataset_manifest.json"

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

scope_df = pd.DataFrame(manifest["scope"]["seasons"])
scope_df

## Step 1 — Check the frozen scope

The table below is our contract. We should not silently add another competition later. A new scope would become a new dataset version.

In [ ]:
print("V1 status:", manifest["status"])
print("Source:", manifest["source"]["provider"])
scope_df[["competition", "season", "expected_matches", "match_360"]]

season = manifest["scope"]["seasons"][0]
season_id = season["season_id"]
match_path = RAW_ROOT / "matches" / str(season["competition_id"]) / f"{season_id}.json"
matches = json.loads(match_path.read_text(encoding="utf-8"))

print("Number of matches:", len(matches))
print("Top-level match keys:", matches[0].keys())
print("First match ID:", matches[0]["match_id"])

In [ ]:
season = manifest["scope"]["seasons"][0]
match_path = RAW_ROOT / "matches" / str(season["competition_id"]) / f"{season["season_id"]}.json"
matches = json.loads(match_path.read_text(encoding="utf-8"))

print("Number of matches:", len(matches))
print("Top-level match keys:", matches[0].keys())
print("First match ID:", matches[0]["match_id"])

### What we are learning

`match_id` is the identifier that connects the match metadata to its event and lineup files. This is the kind of key we will use when building our own relational analytical tables.

In [ ]:
match_id = matches[0]["match_id"]
events = json.loads((RAW_ROOT / "events" / f"{match_id}.json").read_text(encoding="utf-8"))
lineups = json.loads((RAW_ROOT / "lineups" / f"{match_id}.json").read_text(encoding="utf-8"))

print("Events:", len(events))
print("Lineup team records:", len(lineups))
print("First event keys:", events[0].keys())
print("First lineup team keys:", lineups[0].keys())

## Step 3 — Discover event types

Do not decide our features yet. First see what event types actually exist.

In [ ]:
event_types = Counter(
    event.get("type", {}).get("name", "UNKNOWN")
    for event in events
)

pd.Series(event_types).sort_values(ascending=False)

## Step 4 — Inspect football-specific event attributes

Events have a common structure plus type-specific dictionaries. A pass has `pass`; a shot has `shot`; a carry has `carry`, etc. We will only use an attribute after confirming it exists and understanding its meaning.

In [ ]:
pass_event = next(e for e in events if e.get("type", {}).get("name") == "Pass")
shot_event = next(e for e in events if e.get("type", {}).get("name") == "Shot")
carry_event = next(e for e in events if e.get("type", {}).get("name") == "Carry")

print("PASS keys:", pass_event["pass"].keys())
print("SHOT keys:", shot_event["shot"].keys())
print("CARRY keys:", carry_event["carry"].keys())

## Step 5 — Inspect positions

A player can have multiple position intervals in a match because tactical shifts can move the player. We therefore should **not** immediately reduce every player to one permanent position without thinking about the definition.

In [ ]:
position_rows = []
for team in lineups:
    for player in team["lineup"]:
        for position_interval in player.get("positions", []):
            if position_interval.get("position"):
                position_rows.append({
                    "player_id": player["player_id"],
                    "player_name": player["player_name"],
                    "position": position_interval["position"],
                    "from": position_interval.get("from"),
                    "to": position_interval.get("to"),
                })

positions_df = pd.DataFrame(position_rows)
positions_df.head(20)

## Step 6 — Questions we must answer before preprocessing

1. Which events can support attacking, passing, defensive, progression and creativity dimensions?
2. Which metrics should be counts, rates, or per-90 values?
3. What should our minimum-minutes threshold be?
4. How should positions be grouped for peer comparison?
5. Which variables are too sparse or context-dependent to use?

**Do not answer these by intuition alone. We will answer them from the inspection results.**